# GenAI Unified Environment Setup for Google Colab

This notebook sets up a complete Python environment with all necessary dependencies to run all GenAI projects on Google Colab. It handles Google Drive integration, environment configuration, and verification.

**Features:**
- ✅ Installs all required packages in one unified environment
- ✅ Mounts Google Drive for seamless access to your code and data
- ✅ Configures environment variables and API keys
- ✅ Provides utility functions for running code from Google Drive
- ✅ Validates that all dependencies are installed correctly

**Run this notebook first before running any other notebooks from the GenAI project.**

## 1. Install Required Dependencies

Installing all necessary Python packages for the GenAI project including data science, machine learning, LLM, and visualization libraries.

In [ ]:
import subprocess
import sys

# Upgrade pip first
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "-q"])

# List of packages to install
packages = [
    # Core Data Science & ML
    "numpy==2.0.2",
    "pandas==2.2.2",
    "scikit-learn==1.8.0",
    
    # Visualization
    "matplotlib>=3.8.0",
    "seaborn>=0.13.0",
    "altair>=5.0.0",
    "plotly>=5.17.0",
    
    # LLM & GenAI
    "openai==1.93.0",
    "langchain==0.3.26",
    "langchain-openai==0.3.27",
    "langchain-community>=0.3.0",
    "langchainhub==0.1.21",
    "langchain-experimental==0.3.4",
    
    # Web Framework
    "streamlit==1.46.1",
    
    # Environment & Utilities
    "python-dotenv>=1.0.0",
    "requests>=2.31.0",
]

print("Installing packages...")
for package in packages:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"✅ {package}")
    except Exception as e:
        print(f"⚠️  Error installing {package}: {str(e)}")

print("\n✅ All packages installed successfully!")

In [ ]:
# Check if running in Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("✅ Running in Google Colab")
else:
    print("⚠️  Not running in Google Colab. Some features may not work as expected.")
    print("For local testing, you can still run this notebook with local adjustments.")

## 2. Set Up Google Drive Integration

Mount Google Drive to access your uploaded GenAI project files and data.

In [ ]:
import os
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    print("Mounting Google Drive...")
    drive.mount('/content/drive', force_remount=False)
    
    # Set up path to GenAI project
    GENAI_PROJECT_PATH = '/content/drive/My Drive/Projects/GenAI'
    
    # Check if path exists
    if os.path.exists(GENAI_PROJECT_PATH):
        print(f"✅ Found GenAI project at: {GENAI_PROJECT_PATH}")
        os.chdir(GENAI_PROJECT_PATH)
        print(f"✅ Working directory set to: {os.getcwd()}")
    else:
        print(f"⚠️  GenAI project not found at {GENAI_PROJECT_PATH}")
        print("Make sure you've uploaded the GenAI folder to Google Drive under 'My Drive/Projects/'")
else:
    print("Not in Colab. Using local file system.")
    GENAI_PROJECT_PATH = os.getcwd()

print(f"\nProject Path: {GENAI_PROJECT_PATH}")

In [2]:
# GLOBAL SETUP: Configure Python to automatically find your project
import sys
import os
import site

# 1. Define the project path on Google Drive
DRIVE_PROJECT_PATH = '/content/drive/My Drive/Projects/GenAI'

# 2. Locate the site-packages directory in this Colab environment
site_packages = site.getusersitepackages()
if not os.path.exists(site_packages):
    # Fallback to global site-packages if user one doesn't exist
    site_packages = site.getsitepackages()[0]

print(f"📍 Configuring global environment at: {site_packages}")

# 3. Create a 'usercustomize.py' file.
# Python automatically runs this script every time ANY notebook or script starts.
custom_setup_script = f"""
import sys
import os
import site

# The path to your project in Google Drive
project_path = '{DRIVE_PROJECT_PATH}'

# If the path exists (Drive is mounted), add it to sys.path
if os.path.exists(project_path) and project_path not in sys.path:
    sys.path.insert(0, project_path)
    # Also set it as the working directory so relative file loading works
    try:
        os.chdir(project_path)
    except:
        pass
"""

# Write the file
custom_file_path = os.path.join(site_packages, 'usercustomize.py')
try:
    with open(custom_file_path, 'w') as f:
        f.write(custom_setup_script)
    print("✅ GLOBAL CONFIGURATION INSTALLED: 'usercustomize.py' created.")
    print("   Any notebook you open in this session will now automatically find 'utils'.")
except PermissionError:
    print("⚠️  Permission denied writing to site-packages. Creating basic .pth file instead.")
    # Fallback: Create a .pth file which adds the path
    pth_file = os.path.join(site_packages, 'genai_project.pth')
    with open(pth_file, 'w') as f:
        f.write(DRIVE_PROJECT_PATH)
    print("✅ Fallback config installed. Path added to Python.")

# 4. Verify Immediate Effect
if DRIVE_PROJECT_PATH not in sys.path:
    sys.path.insert(0, DRIVE_PROJECT_PATH)

try:
    import utils
    print(f"✅ Immediate verification: 'utils' found at {os.path.dirname(utils.__file__)}")
except ImportError:
    print("⚠️  'utils' not importable yet. It should work in the next notebook you open.")


⚠️ GENAI_PROJECT_PATH not found. Defaulting to current working directory: c:\Projects\GenAI
✅ Added to sys.path: c:\Projects\GenAI


## 3. Initialize Environment Variables and Configuration

Set up environment variables for API keys, file paths, and other configuration settings.

In [ ]:
from dotenv import load_dotenv

# Ensure GENAI project path is available
if 'GENAI_PROJECT_PATH' not in globals():
    print("⚠️  GENAI_PROJECT_PATH is not defined; defaulting to current working directory.")
    import os
    GENAI_PROJECT_PATH = os.getcwd()

# Initialize environment configuration
CONFIG = {
    'project_path': GENAI_PROJECT_PATH,
    'data_path': os.path.join(GENAI_PROJECT_PATH, 'data'),
    'linear_regression_path': os.path.join(GENAI_PROJECT_PATH, 'genAI-foundations', 'machine-learning'),
    'kartify_path': os.path.join(GENAI_PROJECT_PATH, 'genAI-foundations', 'kartify'),
    'python_notebooks_path': os.path.join(GENAI_PROJECT_PATH, 'python'),
    'prework_path': os.path.join(GENAI_PROJECT_PATH, 'pre-work'),
    'in_colab': IN_COLAB,
}

# Try to load .env file if it exists
env_file = os.path.join(GENAI_PROJECT_PATH, '.env')
if os.path.exists(env_file):
    load_dotenv(env_file)
    print("✅ Loaded .env file")
else:
    print("⚠️  No .env file found. API keys must be set manually if needed.")

# Verify critical paths exist
print("\nVerifying paths...")
for key, path in CONFIG.items():
    if key.endswith('_path') and not key.startswith('in_'):
        if os.path.exists(path):
            print(f"✅ {key}: {path}")
        else:
            print(f"⚠️  {key} not found: {path}")

print("\n✅ Environment configuration complete!")
print(f"\nAccess CONFIG dictionary to see all paths:")
print(f"CONFIG['project_path'] = {CONFIG['project_path']}")
print(f"CONFIG['data_path'] = {CONFIG['data_path']}")

## 4. Create Utility Functions for Project Execution

Helper functions to load modules, execute scripts, and manage the environment.

In [ ]:
import importlib.util
from typing import Any, Callable

def load_python_module(file_path: str, module_name: str = None):
    """
    Load a Python module from a file path.
    
    Args:
        file_path: Path to the .py file
        module_name: Name to assign to the module (defaults to filename without extension)
    
    Returns:
        The loaded module
    """
    if module_name is None:
        module_name = os.path.splitext(os.path.basename(file_path))[0]
    
    try:
        spec = importlib.util.spec_from_file_location(module_name, file_path)
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        print(f"✅ Loaded module: {module_name}")
        return module
    except Exception as e:
        print(f"❌ Error loading module {module_name}: {str(e)}")
        return None

def load_csv(file_path: str):
    """Load a CSV file and return as DataFrame."""
    import pandas as pd
    try:
        df = pd.read_csv(file_path)
        print(f"✅ Loaded CSV: {os.path.basename(file_path)} ({len(df)} rows, {len(df.columns)} columns)")
        return df
    except Exception as e:
        print(f"❌ Error loading CSV: {str(e)}")
        return None

def list_project_files(folder_type: str = 'all'):
    """List files in different project folders."""
    folders = {
        'data': CONFIG['data_path'],
        'notebooks': CONFIG['python_notebooks_path'],
        'linear_regression': CONFIG['linear_regression_path'],
        'kartify': CONFIG['kartify_path'],
        'prework': CONFIG['prework_path'],
    }
    
    if folder_type == 'all':
        print("📁 GenAI Project Structure:\n")
        for folder_name, folder_path in folders.items():
            if os.path.exists(folder_path):
                files = os.listdir(folder_path)
                print(f"📂 {folder_name}/ ({len(files)} items)")
                for f in files[:5]:  # Show first 5
                    print(f"   - {f}")
                if len(files) > 5:
                    print(f"   ... and {len(files) - 5} more")
                print()
    else:
        if folder_type in folders and os.path.exists(folders[folder_type]):
            files = os.listdir(folders[folder_type])
            print(f"Files in {folder_type}/:")
            for f in files:
                print(f"  - {f}")

print("✅ Utility functions loaded successfully!")

## 5. Verify Installation and Environment

Validate that all dependencies are installed and the environment is ready.

In [ ]:
import pandas as pd
import numpy as np
import importlib
import sys
import os

print("=" * 60)
print("🔍 ENVIRONMENT VERIFICATION")
print("=" * 60)

# Test critical imports
packages_to_test = {
    'numpy': 'NumPy',
    'pandas': 'Pandas',
    'sklearn': 'Scikit-learn',
    'matplotlib': 'Matplotlib',
    'seaborn': 'Seaborn',
    'altair': 'Altair',
    'plotly': 'Plotly',
    'langchain': 'LangChain',
    'openai': 'OpenAI',
    'dotenv': 'Python-dotenv',
}

failed_imports = []
print("\n✅ Testing core package imports:")
for package_name, display_name in packages_to_test.items():
    try:
        module = importlib.import_module(package_name)
        version = getattr(module, '__version__', 'unknown')
        print(f"  ✅ {display_name}: {version}")
    except ImportError as e:
        print(f"  ❌ {display_name}: Failed to import")
        failed_imports.append(display_name)

print("\n✅ Python Information:")
print(f"  Python Version: {sys.version.split()[0]}")
print(f"  NumPy Version: {np.__version__}")
print(f"  Pandas Version: {pd.__version__}")

# ensure CONFIG has been initialized
if 'CONFIG' not in globals():
    print("⚠️  CONFIG not found. Creating a default configuration based on the current working directory.")
    CONFIG = {
        'project_path': os.getcwd(),
        'data_path': os.path.join(os.getcwd(), 'data'),
        'linear_regression_path': os.path.join(os.getcwd(), 'genAI-foundations', 'machine-learning'),
        'kartify_path': os.path.join(os.getcwd(), 'genAI-foundations', 'kartify'),
        'python_notebooks_path': os.path.join(os.getcwd(), 'python'),
        'prework_path': os.path.join(os.getcwd(), 'pre-work'),
        'in_colab': IN_COLAB if 'IN_COLAB' in globals() else False,
    }
    print("✅ Default CONFIG dictionary created. Consider running the full environment configuration cell later for more comprehensive setup.")

print("\n✅ Project Configuration:")
print(f"  In Colab: {CONFIG['in_colab']}")
print(f"  Working Directory: {os.getcwd()}")
print(f"  Project Path: {CONFIG['project_path']}")

# List project contents
print("\n✅ Project Contents Preview:")
if 'list_project_files' not in globals():
    # stub to avoid NameError
    def list_project_files(*args, **kwargs):
        print("⚠️  list_project_files() is not yet defined. Run the utility cell (step 4) to get the full implementation.")

list_project_files()

if failed_imports:
    print(f"\n⚠️  Warning: {len(failed_imports)} packages failed to import: {', '.join(failed_imports)}")
    print("Try reinstalling failed packages if needed.")
else:
    print("\n" + "=" * 60)
    print("✅ ALL CHECKS PASSED - ENVIRONMENT READY!")
    print("=" * 60)
    print("\nYou can now:")
    print("  1. Run notebooks from genAI/genAI-foundations/machine-learning/")
    print("  2. Run notebooks from genAI/python/")
    print("  3. Use the load_csv() function to load data files")
    print("  4. Use load_python_module() to import Python scripts")
    print("\nExample: df = load_csv(CONFIG['data_path'] + '/Sales.csv')")
    print("\nNext Steps: Open any notebook from the project and start coding!")